
# 1. Before we start



## 🕵️ Messy by Default
### A Hands-On Look at Real Data

This workshop is about the gap between raw data and a data story, and how that gap gets closed — carefully or not.

Every dashboard, chart, or model starts long before anyone opens a design tool. It starts with raw, messy data and a series of quiet decisions: what to keep, what to drop, how to group, how to fill in the gaps. Those decisions shape the story just as much as the final visual does.

We'll start with the basics: what data science actually is, how it differs from data analysis, and what a data scientist's day looks like compared to a data analyst's. Data science covers different subfields and approaches, from simple averages to full models; we'll touch on a few. Along the way, a short glossary of terms you'll keep hearing: features, correlation, model, ground truth.

Then, hands-on: we'll work with a small dataset of real house prices, try a few basic techniques together, and build two technically accurate dashboards from the same data that still tell two different stories.




<p align="center">
  <img src="images/what_is_data_science.png" alt="Sticky-note answers to 'What is data science?' plus a short definition" width="800">
</p>

*Several ways to say the same job: turn messy data into something someone can act on — often a model, a recommendation, or a clearer question.*

<p align="center">
  <img src="images/data_science_flow.png" alt="Data pipeline from define objective through ingest, clean, understand, transform, analyze/model, deploy, and visualize" width="600">
</p>

*The tidy version of the job. Notice that **Visualize** sits last — that's the official story. Today we'll see why chart choices actually leak into every earlier step.*


## Who does what? 

| Role | What they actually do | Goal |
|---|---|---|
|🧑🏽‍🔧**Data Engineer** | Build stable pipelines, data architectures, and databases that stay reliable. | *how do we move and store this data reliably?* |
|🕵🏻‍♀️**Data Analyst** | Explore existing data to answer specific questions and visualize the findings. | *what happened, and why?* |
|🧑‍🔬**Data Scientist** | Build models to predict, and design the experiments and pipelines to test those models. More statistics, more engineering. | *what will happen?* |
| 🤖 **ML Engineer** | Take a working model and make it run reliably in production: serving, monitoring, retraining. | *how do we keep this prediction running?* |

Today's fun sits mostly in **Data Analyst** territory, with a small **Data Scientist**-style taste at the end (chapter 7).



<p align="center">
  <img src="images/data_roles.png" alt="Which roles typically cover which steps of the data pipeline" width="600">
</p>

*Roles overlap on purpose. Data scientists often span the whole pipeline; ML engineers concentrate on the later "make it run" steps. Today's work lives mainly in Clean → Understand → Analyze.*

### A few terms in Data Science

| Term | Meaning |
|---|---|
| **Feature / input** | A column used to explain or predict something else (`area`, `rooms` → `price`) |
| **Correlation** | Two things moving together. Strong correlation is not the same as one causing the other. |
| **Outlier** | A point that sits far from the rest. It might be an error, or a real but extreme case — those two need different treatment. |
| **Model** | A rule (often fitted from data) that turns features into a prediction |
| **Target / output** | The thing you're trying to predict (here: `price`) |
| **Ground truth / label** | The "actual" correct value you compare predictions or cleaning decisions against. Often more assumed than truly known. |


**What we'll do:**
1. Before we start — what data science is, who does what
2. 📦 Set up, load the data, and clean it
3. 🔎 One street, and what counts as an outlier
4. 📊 Explore correlations: what really moves with housing price?
5. 🗺️ Put the prices on a map
6. 🎭 Build two honest dashboards that tell two different stories
7. 🎁 *(Optional, for the curious)* a tiny price-prediction model
8. 💬 Discuss: what did we just do, and where does this happen in real projects?

> 💡 **How to use this notebook:** The eight **chapters** are the `#` headings. Click the fold arrow next to a chapter title to close that chapter and jump around. Every section has runnable code, but you don't need to write code to participate. Read the markdown, look at the charts, and jump into the discussion. Cells marked **🎯 Optional / Bonus** are for anyone who wants to go further; feel free to skip them and stay in the conversation instead.



# 2. Setup, ingest, and cleaning 📦



## 1. Setup 📦

Run the cell below to load the libraries we'll use. Nothing exotic: `pandas` for data handling and `matplotlib`/`seaborn` for charts.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make charts a bit bigger and cleaner by default
plt.rcParams["figure.figsize"] = (8, 5)
sns.set_style("whitegrid")

pd.set_option("display.max_columns", None)
print("Libraries loaded ✅")


## The brief: the question we *think* we're answering 🏠


> **I want to buy a property in Amsterdam. What actually drives the price?**

Hold onto this. We'll load the data first, then come back to why this brief is the wrong shape for this dataset. 





## 2. Ingest: what are we even looking at? 📥

We collected a scrape of Funda house sales. Let's load the data and take a first look — before we clean anything, before we assume anything.

**What do you expect a housing dataset to contain?**  
**What could already be messy, or even wrong, in this dataset?**


In [ ]:

df = pd.read_csv("funda.csv")

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head()


In [ ]:
df = df.drop(columns=["url"])
df.sample(5)

In [ ]:
# Get the actual unique values for each column
print("\n---  Unique Values ---")
df["posting_year"]= pd.to_datetime(df["posting_date"], format="%d-%m-%Y").dt.year

for col in ["property_type", "bedrooms", "rooms", "posting_year"]:
    print(f"{col}: {np.sort(df[col].unique())}\n")


**Why the question is the wrong shape**

1.  **Wrong year.** These are mostly sold Funda listings from **2015–2016** (a few posts go back to 2009). You cannot use them to decide what to offer in 2026. Interest rates, transfer tax, COVID, and everything after 2016 are missing.
2. **Wrong decision.** “I want to buy” needs today’s asking prices, one neighbourhood, one budget, one type. This scrape knows what sold *then*, not what you should pay *now*.
3. **Wrong promise: “what drives.”** Drive sounds like cause. At best we will see associations: bigger homes cost more; older, central streets often cost more per m².

So we keep the curiosity and change the brief:

**We cannot tell you what to pay for a home in Amsterdam in 2026.**  
**We can show how someone *would* have answered that buying question from this scrape — and how easy it is to overclaim.**


**A first pass with `.info()` and `.describe()`** — this is usually the very first thing a data analyst does with any new dataset. It won't tell you if the data is *right*, only what *shape* it's in.


In [ ]:
# could you use info to check for multiple data types in the same column?
df.info()


In [ ]:

df.describe()



### 🔍 Look closely at that `.describe()` table. Anything look off?

A few things worth noticing already, just from summary statistics:
- `price` ranges from **€1,000** to **€999,999** — is a €1,000 apartment in Amsterdam plausible?
- `year_built` has a **minimum around 1005** — Amsterdam wasn't founded until roughly 1275.
- `area` has a max of over 800 m² — rare, but possible for the right property.

None of these are proven mistakes yet. That's the point: **`.describe()` gives you suspects, not verdicts.** Let's investigate.



## 3. The detective work: exploring before cleaning 🕵️

### 3.1 Missing values

Let's start with the easy check.


In [ ]:
# What could go wrong with this missing value check?
print(df.isna().sum())


In [ ]:
import numpy as np

# List of values to consider as missing
suspicious_missing = ["", "unk",  "unknown", "Unknown", "UNKNOWN", 0, -1, "NA", "na", "N/A"]

# For each column, count how many suspicious/missing codes are present
for col in df.columns:
    vals = df[col].values
    mask = np.isin(vals, suspicious_missing)
    suspicious_count = mask.sum()
    if suspicious_count > 0:
        print(f"Column '{col}' has {suspicious_count} suspicious/missing value(s)")

# Optionally, replace all such with np.nan for true missing value handling
# df = df.replace(suspicious_missing, np.nan)



Good news: no missing values here. But **"no nulls" does not mean "no problems"** — a wrong number is often worse than a missing one, because it doesn't announce itself. 


### 3.2 Duplicates: check before you trust any number


In [ ]:
print("identical rows:", int(df.duplicated().sum()))
print("repeat addresses:", int(df["address"].duplicated().sum()))
print("same address + price + area:", int(df.duplicated(["address", "price", "area"]).sum()))
print("same address + sale day:", int(df.duplicated(["address", "sale_date"]).sum()))

# the actual listings (not just the count)
dups = (
    df[df.duplicated("address", keep=False)]
    .sort_values(["address", "sale_date"])
)
#dups[["address", "price", "area", "sale_date", "posting_date", "url"]]

In [ ]:
same_day = (
    df[df.duplicated(["address", "sale_date"], keep=False)]
    .sort_values(["address", "sale_date"])
)
same_day[["address", "price", "area", "sale_date"]]

A scrape rarely copies the same row twice. What it *does* do is catch the **same home more than once**: relisted, new Funda URL, same address. `id` and `url` can all be unique and you still have double sales.

That quietly **skews everything downstream**:
- counts (“11,443 sales”) are too high
- averages and medians get pulled by the same €545k flat counted three times
- “how many homes sold on this street?” is wrong
- a model sees extra identical examples and gets overconfident

A relist is not a second sale; a resale a year later might be.


### 3.3 The suspiciously cheap listings 💸

Let's look at the cheapest sales in the dataset.


In [ ]:

cheapest = df.nsmallest(10, "price")[["address", "area", "bedrooms", "price", "property_type", "year_built"]]
cheapest



**Discussion:** A 222 m² apartment for €1,000? A 277 m² place on Sophialaan (one of Amsterdam's most expensive streets) for €1,000?

These are almost certainly **not real market prices** — likely family transfers, data entry placeholders, or scraping errors where the real price wasn't captured. If we don't catch this, every average, chart, and prediction downstream will be quietly wrong.

Let's visualize the price distribution to see how big this problem is.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df["price"], bins=60, ax=axes[0], color="#4C72B0")
axes[0].set_title("Price distribution (raw)")
axes[0].set_xlabel("Price (€)")

# log_scale=(False, True) draws bars from y=0, which a log axis cannot show.
# Plot first, then set a log y-axis that starts at 1.
sns.histplot(df["price"], bins=60, ax=axes[1], color="#E07A3D")
axes[1].set_yscale("log")
axes[1].set_ylim(bottom=1)
axes[1].set_title("Same data, log y-axis\n(reveals the low-price spike)")
axes[1].set_xlabel("Price (€)")


plt.tight_layout()
plt.show()



### Why a log scale, and how it works

A **linear** y-axis treats every extra listing the same: 10 more sales and 1,000 more sales take the same amount of extra height. That sounds fair, but it hides the interesting part. In this dataset most homes sit in a big pile around a few hundred thousand euros, so that pile uses almost all the vertical space. The €1,000 cluster becomes a thin smudge against the axis.

A **log** (logarithmic) y-axis changes the rule from "add" to "multiply." Each step up the axis is ×10, not +10:

| Count | Linear height | Log₁₀ height |
|---|---|---|
| 1 | 1 | 0 |
| 10 | 10 | 1 |
| 100 | 100 | 2 |
| 1,000 | 1,000 | 3 |
| 10,000 | 10,000 | 4 |

So a bin with 10 listings and a bin with 10,000 listings can both be *seen*. That is why the middle chart suddenly shows the cheap-price spike: those rows were always there; the linear chart just did not give them any ink.

Two caveats worth keeping in your pocket:

1. **Log cannot show zero.** `log(0)` is undefined, which is why the middle chart's y-axis starts at 1, not 0.
2. **Log is a storytelling choice.** It makes rare groups more visible. That can be honest (here: "look, fake €1,000 sales") or misleading (making a handful of rows look as important as thousands). Same data, different emphasis — the theme of this workshop.

House prices, income, population, and website hits are classic log-axis candidates. Year built, room count, and a 0–100 score usually are not.

> 🎯 **Optional / Bonus (for the machine-learning-curious):** skip this if you want to stay with the charts. A log *axis* (what we just did) leaves `price` untouched. A log *transformation* is a different move: you create a new column, e.g. `df["log_price"] = np.log10(df["price"])`, and a model trains on those numbers. That compresses a long right tail — one €5 million house no longer sits miles from the €300k cluster — so a linear model is less pulled around by extreme wealth. Two things not to mix up: (1) this is still not cleaning; fake €1,000 sales should be dropped or flagged first, because log only rescales them; (2) the model now answers “X% more,” not “€X more.” We come back to a tiny price model in chapter 7.



> 🎨 **Design note:** the middle chart shows the *exact same data* as the one on the left — the only change is a log scale on the y-axis. Notice how much easier the low-price cluster is to spot. The third chart uses `df_clean` (duplicates and ≤ €10k prices removed); `df` is still the raw table, so you can keep comparing both. This is your first taste of today's later exercise: **the same honest data can look completely different depending on how you choose to show it.**

### 3.4 The impossible construction years 🏚️

Now the `year_built` issue.


In [ ]:

oldest = df.nsmallest(10, "year_built")[["address", "year_built", "price", "area"]]
oldest



Amsterdam does have real 17th-century canal houses (`Herengracht`, `Warmoesstraat` — those ~1600s values are plausible!). But **1005** and **1076** are not — those are almost certainly typos, likely `1905` and `1976` with a digit dropped or swapped.

**This is an important distinction for any data cleaning decision:** not every outlier is an error, and not every error looks extreme. A silent typo (`1905` → `1005`) is a real risk in any dataset your team touches — spreadsheets, CRM exports, manually entered forms.

### 3.5 Deciding what to do about it

There's no single "correct" answer here — only trade-offs. The useful first question is: **is this an error, or a real but extreme value?** €1,000 on Sophialaan is an error. A real €2 million canal house is extreme. Those two ask for different moves.

| Approach | What it does | Who might prefer it, and why |
|---|---|---|
| **Drop the rows** | Remove listings with `price < €10,000` or `year_built < 1200` | A developer building a pipeline — simple, defensible, reproducible. Also a common first move when the number is an *error* (you do not want a model trained on a fake sale). |
| **Cap / clip the values** | Keep the row, but replace the extreme number with a floor or ceiling. Example: every price below €10,000 becomes €10,000. Area and address stay. | Tempting if you want to keep the rest of the row. Easy to confuse with dropping: the listing is still there, but you have written a price that never happened. A reasonable tool for *real* extremes (a few penthouses pulling an average). A poor fit here — €1,000 → €10,000 is still not the sale price, and year `1005` → `1200` invents a medieval house. |
| **Flag, don't touch** | Leave the data as-is but add an `is_suspicious` column | A designer building a dashboard — lets the *end user* see and decide, rather than hiding the decision |
| **Investigate further** | Go back to the source (Funda listing URL) to check if it's real | The most rigorous option — rarely done in practice due to time |
| **Transform or try both** | For real skew: model `log(price)`, or run the analysis once with the extremes and once without, and see if the story changes | A data scientist — they care whether a prediction gets *pulled by the tail*. They still drop or flag clear errors first; a log does not turn €1,000 into a real sale. |

**For this workshop, we'll drop the clearest errors** so our later charts aren't distorted by a handful of bad rows — but keep in mind this is a *choice*, not a neutral default.


In [ ]:

before = len(df)

df_clean = df.drop_duplicates(subset=["address", "price", "area"]).copy()
df_clean = df_clean[
    (df_clean["price"] > 10_000) &
    (df_clean["year_built"] > 1200) &
    (df_clean["year_built"] <= 2016)  # dataset appears to run through 2016
]

after = len(df_clean)
print(f"Removed {before - after} rows ({(before - after) / before:.1%} of the dataset)")
print(f"Remaining: {after:,} rows")



> 🎯 **Optional / Bonus:** try changing the thresholds above (e.g. `price > 50_000`, or `year_built > 1500` to keep the real canal houses) and re-run. Watch how many rows get removed each time — small threshold decisions can meaningfully shift your dataset size and, later, your conclusions.

### 3.6 One more feature worth creating: price per m²

Raw price alone conflates "expensive" with "big." A more useful comparison is **price per square meter** — the actual metric used in real estate analysis.


In [ ]:

df_clean["price_per_m2"] = df_clean["price"] / df_clean["area"]
df_clean[["address", "area", "price", "price_per_m2"]].sort_values("price_per_m2", ascending=False).head()



# 3. One street, and what counts as an outlier 🔎


## 🔎 Case study: one street, many prices — Amstelkade

A useful question when the whole city is too noisy: **hold location (almost) still, and ask why prices still differ.**

`Amstelkade` is a good slice for that. There are a dozen sales on the same street, all apartments, all built 1906–1930. Prices still run from about **€230k to €815k**. Is that "some houses are just nicer," or is it mostly size, floor, and a few other columns we already have?

> 💡 This is a *specific* question, and that's the point. "What drives price in Amsterdam?" is huge. "On this one street, why is 113 more expensive than 50-3?" is something you can actually check in ten minutes.


In [ ]:
amstel = (
    df_clean[df_clean["address"].str.contains("Amstelkade", case=False)]
    .sort_values("price")
    .copy()
)

amstel["sale_year"] = pd.to_datetime(amstel["sale_date"], format="%d-%m-%Y").dt.year

print(f"{len(amstel)} Amstelkade sales  |  price €{amstel['price'].min():,.0f}–€{amstel['price'].max():,.0f}")
#print(f"Median €/m² on this street: €{amstel['price_per_m2'].median():,.0f}")
#print(f"Median €/m² in the cleaned city-wide set: €{df_clean['price_per_m2'].median():,.0f}")

amstel[["address", "area", "bedrooms", "rooms", "price", "price_per_m2", "year_built", "sale_year"]]


In [ ]:
# This code raises "ValueError: Could not interpret value `sale_year` for `hue`" 
# because the 'sale_year' column does not exist in the amstel DataFrame at runtime,
# likely due to a failure in the previous cell or a re-run order issue.

# To debug, check if 'sale_year' is present:
print(amstel.columns)

# Explanation:
# The error "Could not interpret value `sale_year` for `hue`" means your DataFrame 'amstel' 
# does not have a column called 'sale_year' at this point, so Seaborn cannot color points by that column.

# Example fix - create 'sale_year' if missing:
if "sale_year" not in amstel.columns:
    amstel["sale_year"] = pd.to_datetime(amstel["sale_date"], format="%d-%m-%Y").dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
year_colors = {2015: "#4C72B0", 2016: "#E07A3D"}

sns.regplot(data=amstel, x="area", y="price", scatter=False, ax=axes[0], color="0.7", ci=None)
sns.scatterplot(
    data=amstel, x="area", y="price",
    hue="sale_year", palette=year_colors, s=90, ax=axes[0],
)
for _, row in amstel.iterrows():
    axes[0].annotate(
        row["address"].replace("Amstelkade ", ""),
        (row["area"], row["price"]),
        textcoords="offset points", xytext=(6, 4), fontsize=8,
    )
axes[0].set_title("Price vs. size\n(colour = sale year)")
axes[0].set_xlabel("Area (m²)")
axes[0].set_ylabel("Price (€)")

sns.scatterplot(
    data=amstel, x="area", y="price_per_m2",
    hue="sale_year", palette=year_colors, s=90, ax=axes[1], legend=False,
)
for _, row in amstel.iterrows():
    axes[1].annotate(
        row["address"].replace("Amstelkade ", ""),
        (row["area"], row["price_per_m2"]),
        textcoords="offset points", xytext=(6, 4), fontsize=8,
    )
axes[1].set_title("€/m² vs. size\n(113 sits above the rest)")
axes[1].set_xlabel("Area (m²)")
axes[1].set_ylabel("Price per m² (€)")

plt.tight_layout()
plt.show()



## Spotting a true outlier

<p align="center">
    <img src="images/outlier.png" alt="A cluster of typical points with two outliers sitting far from the rest" width="500"/>
</p>

*Not every far-away point is a mistake — and not every expensive home is an outlier. The next bullets use this street to tell those two apart.*

- In most cases, price still closely follows **area**—bigger apartments almost always cost more, following the overall grey trend line.
- Not every "expensive" home is an outlier: a place like 65 D, while the priciest, simply earns its price by being the largest (and its €/m² isn't extraordinary). High price, but expected given its size.
- By contrast, some homes are *true outliers*: for example, Amstelkade 113. It's not the biggest, but its total price is near the top, and its **€/m² is much higher** than its neighbors—standing well apart from the street's usual size–price pattern. This isn't a data error; it's a genuine standout — perhaps due to a unique renovation, a prime location, or some hidden feature the data doesn't tell us.

On this street we *looked* at the size–price line so we could see who sits off it. That is still about a dozen homes, not Amsterdam.

> 🎯 **Optional / Bonus:** Try this analysis for another street with `df_clean["address"].str.contains("…")` (for example, `Sarphatipark` or `Herengracht`). Do you spot true outliers there, or does price smoothly track area?



# 4. Correlation 📊


## What actually correlates with price?

<p align="center">
    <img src="images/correlation.png" alt="Scatterplot showing a high degree of positive correlation: as x increases, y increases" width="400"/>
</p>

*Two variables moving together. We'll measure that next — and then remind ourselves it still isn't cause.*

Now zoom back out. The heatmap uses the **full cleaned dataset** — every row we kept, not Amstelkade. Same idea (what moves with price?), different scale: a number for every pair of columns, not a story about one address.

**Before running the next cell, take a moment to consider:**  
What do you *expect* will have a strong correlation with price? Area? Number of rooms? Age of the building?


In [ ]:

numeric_cols = ["area", "bedrooms", "rooms", "price", "year_built", "price_per_m2"]
corr = df_clean[numeric_cols].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()



A few things worth pointing out once you see the heatmap:

- `area` and `price` are usually the strongest relationship — bigger places cost more, unsurprisingly.
- `bedrooms` and `rooms` correlate strongly *with each other* (as expected — they're not independent features), and both correlate less strongly with price than `area` does. This is worth sitting with: two features can each seem "important" individually while mostly just measuring the same underlying thing (size).
- `year_built`'s correlation with price is usually much weaker than people expect. Amsterdam's housing market doesn't reward "newer" the way people might assume — a beautifully located 1900s canal house can outprice a modern build.

> ⚠️ **The one line every data-adjacent person should say out loud at least once a year:**
> **Correlation is not causation.** A strong correlation between `area` and `price` doesn't prove that adding square meters *causes* a price increase by some fixed amount — location, property type, and market timing are tangled in there too.

### 4.1 Visualizing the strongest relationship


In [ ]:

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_clean, x="area", y="price",
    hue="property_type", alpha=0.4, s=25
)
plt.title("Price vs. Area, by property type")
plt.xlabel("Area (m²)")
plt.ylabel("Price (€)")
plt.tight_layout()
plt.show()



> 🎯 **Optional / Bonus:** try re-coloring the scatterplot by a different column, e.g. `bedrooms` or the first 4 digits of `postal_code` (you'll need to extract it first: `df_clean["postal_code"].str[:4]`). Does a location pattern emerge?


# 5. Maps 🗺️


In [ ]:
postcode_df = pd.read_csv("Nederland_postcodes.csv")
postcode_df.head()

In [ ]:
df_clean['postal_code_clean'] = df_clean['postal_code'].str.replace(' ', '', regex=False).str.upper()
postcode_df['postcode_clean'] = postcode_df['postcode'].str.replace(' ', '', regex=False).str.upper()

df_clean = df_clean.merge(
    postcode_df[['postcode_clean', 'lat', 'lon']],
    left_on='postal_code_clean',
    right_on='postcode_clean',
    how='left'
)

df_clean = df_clean.drop(columns=['postal_code_clean', 'postcode_clean'])
df_clean.head()

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

plot_df = df_clean.dropna(subset=['lat', 'lon'])

blue_ramp = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
    "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
    "#184f95", "#104281", "#0d366b",
]

fig = px.scatter_map(
    plot_df,
    lat='lat',
    lon='lon',
    color='price',
    color_continuous_scale=blue_ramp,
    hover_name='address',
    hover_data={'price': True, 'area': True, 'lat': True, 'lon': True},
    center={'lat': 52.37, 'lon': 4.895},
    zoom=10.5,
    height=700,
)

fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.update_layout(
    map_style='carto-positron',
    margin={'r': 0, 't': 0, 'l': 0, 'b': 0},
    coloraxis_colorbar=dict(title='Price (€)'),
)

fig



# 6. Dashboards 🎭


Here's the core exercise. Using the **exact same cleaned dataset**, we're going to build two small "dashboards" (a couple of charts each) — one that tells an **optimistic, reassuring** story about the Amsterdam housing market, and one that tells a **cautious, concerning** one.

**The rule: every chart must be technically accurate.** No fabricated numbers, no mislabeled axes. Just different, equally legitimate choices about:
- What time period or subset to show (**snapshot vs. change**)
- How to bin or group the data
- Which metric to lead with (mean vs. median, total vs. per m²)
- What to zoom in on vs. leave out

One shared honest cut, first: posting years **2009–2013 have only 7–85 listings each**. Drawn as equal bars next to 2015's 6,000+ sales, they look like "the 2009 market." They are not — they are leftover listings that took years to sell. Both dashboards stick to the well-sampled years. The spin starts after that.

### 5.1 Dashboard A: "The market is healthy and accessible" 🟢

A *trend* of rising prices will not look accessible, however green you paint it. So this dashboard does not plot a trend. It shows a **2016 snapshot of apartments** — the cheaper property type, in the latest well-sampled year — and asks how the typical listing looks.

*(This uses `posting_date`, so let's first parse it into an actual date.)*


In [ ]:

df_clean["posting_date_parsed"] = pd.to_datetime(df_clean["posting_date"], format="%d-%m-%Y")
df_clean["posting_year"] = df_clean["posting_date_parsed"].dt.year

# Snapshot: 2016 apartments only — cheaper segment, latest well-sampled year
apt_2016 = df_clean[
    (df_clean["posting_year"] == 2016) & (df_clean["property_type"] == "apartment")
].copy()
median_price = apt_2016["price"].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: where 2016 apartment prices sat (a pile, not a time series)
sns.histplot(apt_2016["price"], bins=40, ax=axes[0], color="#2a9d8f")
axes[0].axvline(median_price, color="0.2", linestyle="--", linewidth=1.5,
                label=f"Median €{median_price:,.0f}")
axes[0].axvline(300_000, color="0.35", linestyle=":", linewidth=1.5, label="€300k")
axes[0].set_xlim(0, 700_000)  # the right tail still exists; it just isn't in frame
axes[0].set_title("Most 2016 apartments clustered below €300k\n(the typical listing looks affordable)")
axes[0].set_xlabel("Asking price (€)")
axes[0].set_ylabel("Number of apartments")
axes[0].legend()

# Chart 2: majority-under-€300k as a 2016 snapshot (not the declining yearly line)
bands = pd.cut(
    apt_2016["price"],
    bins=[0, 300_000, 400_000, np.inf],
    labels=["€300k or below", "€300–400k", "Over €400k"],
)
band_share = (
    bands.value_counts(normalize=True)
    .reindex(["€300k or below", "€300–400k", "Over €400k"]) * 100
)
band_share.plot(kind="bar", ax=axes[1], color=["#2a9d8f", "#74c69d", "#b7e4c7"], rot=0)
axes[1].set_ylim(0, 100)
axes[1].set_title("In 2016, a majority of apartments\nwere still listed at €300k or below")
axes[1].set_ylabel("% of 2016 apartment listings")
axes[1].set_xlabel("")
for i, value in enumerate(band_share):
    axes[1].text(i, value + 2, f"{value:.0f}%", ha="center")

plt.tight_layout()
plt.show()



> **What's the trick?** These numbers are real (2016 apartments: median €275k; ~61% at €300k or below). The spin is *what is left out*. This is one year, not 2014→2016. Apartments only, not houses. A histogram cropped at €700k, so the expensive tail is off-stage. And a **snapshot** of "still a majority" instead of the yearly line — that share was ~80% in 2014, and plotting it by year would wreck the accessible story.

### 5.2 Dashboard B: "The market is overheating and squeezing buyers out" 🔴

Same underlying data. Now we show **change** over the well-sampled years, in a harsher unit, with the axis zoomed so the jump fills the frame.


In [ ]:

recent = df_clean[df_clean["posting_year"].between(2014, 2016)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: even the median €/m² is harsh once you look at change
ppm2_by_year = recent.groupby("posting_year")["price_per_m2"].median()
ppm2_by_year.index = ppm2_by_year.index.astype(int)
ppm2_by_year.plot(kind="bar", ax=axes[0], color="#e76f51", rot=0)
axes[0].set_ylim(2000, 5000)  # zoom: €0 is the true floor; we start at €2,000
axes[0].set_title("Median price per m², 2014–2016\n(+66% in two years)")
axes[0].set_ylabel("€ per m²")
axes[0].set_xlabel("Year")
for i, value in enumerate(ppm2_by_year):
    axes[0].text(i, value + 60, f"€{value:,.0f}", ha="center", fontsize=9)

# Chart 2: share over €400k, with a cropped y-axis so the doubling looks big
expensive_share = (
    recent.assign(is_expensive=lambda d: d["price"] > 400_000)
    .groupby("posting_year")["is_expensive"].mean() * 100
)
expensive_share.index = expensive_share.index.astype(int)
expensive_share.plot(kind="line", marker="o", ax=axes[1], color="#e76f51")
axes[1].set_xticks(expensive_share.index)
axes[1].set_ylim(0, 30)  # cropped: 24% now fills most of the chart
axes[1].set_title("Share of ALL listings priced over €400k\n(nearly doubled)")
axes[1].set_ylabel("% of listings")
axes[1].set_xlabel("Year")

plt.tight_layout()
plt.show()



> **What's the trick?** Also real (median €/m² €2,593 → €4,310; share over €400k ~13% → ~24%). The spin is *what is emphasized*. This is a **two-year change**, not a 2016 level. All property types, not apartments. €/m² instead of total price (homes also got a bit smaller, so this unit rises faster). The left axis starts at €2,000, not €0. The right axis stops at 30%, so a still-minority of listings fills the frame. "Nearly doubled" is a relative change: 24% is not most of the market.

### 💬 Discuss as a group

- Both dashboards are built from **the same cleaned dataset**, with **no invented numbers**. Yet they leave very different impressions.
- Which one would a real estate agency want to show buyers? Which one would a tenants' rights group want to show?
- Look back at the choices that actually created the difference this time: **snapshot vs. change**, apartments-only vs. all types, total price vs. €/m², "under €300k" vs. "over €400k", cropped axes, a histogram that stops before the tail. Which of these would *you* have made without thinking twice?
- What happens to Dashboard A if you plot the under-€300k share **by year** instead of the 2016 snapshot? (Try it — the accessible story falls apart.)
- **For the designers in the room:** which chart *design* choices (color, framing, cropped axes, what sits inside the plot window) reinforce each dashboard's story on top of the number choices?

> 🎯 **Optional / Bonus:** build a third dashboard — the *most neutral, least persuasive* version you can make of the same data. Is it actually possible to be neutral, or does every choice still lean somewhere?



# 7. Machine learning 🎁


This section is entirely optional — skip it and stay in discussion if you'd rather. For anyone curious what a *very* simple version of "data science" (as opposed to data analysis) looks like, here's a minimal model that predicts price from a few features.

The glossary at the top still applies. In pictures:

<p align="center">
  <img src="images/input_model_output.drawio.svg" alt="A model turns input features into an output prediction">
</p>

<p align="center">
  <img src="images/model_ground_truth.drawio.svg" alt="A model learns parameters by comparing its output to ground-truth labels">
</p>

*Features go in; a prediction comes out. The model improves by comparing that prediction to **ground truth** — here, the sale prices in the table, which we treat as correct even though we know some rows were wrong before cleaning.*

**Important framing:** this is a toy example to illustrate the idea, not a production-quality model. We are deliberately **not** doing the deeper work (train/test splits done properly, feature engineering, handling categorical variables carefully, model validation) a real project would need — that's a whole separate workshop.


In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

features = ["area", "bedrooms", "rooms", "year_built"]
X = df_clean[features]
y = df_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Model coefficients (how much each feature moves the predicted price):")
for feature, coef in zip(features, model.coef_):
    print(f"  {feature:12s}: €{coef:,.0f} per unit")

print(f"\nMean Absolute Error: €{mean_absolute_error(y_test, predictions):,.0f}")
print(f"R² score: {r2_score(y_test, predictions):.2f}  (1.0 = perfect, 0.0 = no better than guessing the average)")



### How wrong is the model, visually?


In [ ]:

plt.figure(figsize=(7, 6))
plt.scatter(y_test, predictions, alpha=0.4, s=25)
lims = [0, max(y_test.max(), predictions.max())]
plt.plot(lims, lims, color="black", linestyle="--", label="Perfect prediction")
plt.xlabel("Actual price (€)")
plt.ylabel("Predicted price (€)")
plt.title("Predicted vs. Actual price")
plt.legend()
plt.tight_layout()
plt.show()



**Discussion:** notice how far some points sit from the dashed "perfect prediction" line — those are listings this simple model badly misjudges. A real data scientist's job is largely about narrowing that spread: better features, better models, better validation. Today's version is intentionally the "quick sketch," not the finished building.

> 🎯 **Optional / Bonus:** add `property_type` as a feature (you'll need to convert it to a number first, e.g. `pd.get_dummies(df_clean["property_type"])`) and see if the R² score improves.



# 8. Wrap-up 💬


This notebook took us through the real mess of a housing scrape: fake-looking €1,000 listings, impossible construction years, the same home listed more than once, and a street where "expensive" was not the same as "outlier." We then zoomed back out: a correlation on the full cleaned table is a different claim from a story about a dozen homes. We also saw that **"no missing values" does not mean "no problems."** Working with data means a lot of detective work **before** any machine learning or visualization.

Some honest lessons to carry forward:

- **`.info()` and `.describe()` are a starting point, not a diagnosis.** You need to dig into suspicious values and rows; the summary only hints at deeper issues.
- **Cleaning choices are modeling choices.** Each time you drop a row, impute a value, or cap an outlier, you're making a call that impacts what your audience will believe about the data.
- **Transforming and visualizing data opens possibilities, not just pictures.** Merging in the postcode table and visualizing the data on a map didn't just make things prettier, it surfaced different patterns and felt more intuitive for different questions, which is a reminder that there's rarely one "right" way to show your data.
- **A street and a heatmap answer different questions.** Twelve homes on Amstelkade show who sits off the line. The correlation matrix uses the whole cleaned dataset. Same idea (what moves with price?), different scale.
- **The same dataset can support wildly different stories.** The metrics and groups you choose can lead to different, but still true, narratives — which is why ethical and clear communication is so important.
- **Analyst vs. scientist:** Today our work was mostly analyst (exploring, visualizing, cleaning), with a final taste of data scientist (building and evaluating a predictive model).

Thanks for working through this hands-on, messy data journey with us. We hope you'll bring a more skeptical eye, and practical cleaning habits, to your own projects. 🏡📊
